In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
pd.set_option("display.max_info_columns", 150)
import numpy as np

from src.database.connection import get_connection

import warnings
warnings.filterwarnings(
    "ignore",
    message = "pandas only supports SQLAlchemy connectable.*",
    category = UserWarning
)

#### LOAD DATA

In [2]:
conn = get_connection()

query = """
    SELECT *
    FROM generation_eda
    ORDER BY start_time;
"""

generation = pd.read_sql(query, conn)
generation["publish_time"] = pd.to_datetime(generation["publish_time"], utc = True)
generation["start_time"] = pd.to_datetime(generation["start_time"], utc = True)

query = """
    SELECT *
    FROM weather_eda
    ORDER BY forecast_time;
"""

weather = pd.read_sql(query, conn)
weather["forecast_time"] = pd.to_datetime(weather["forecast_time"], utc = True)

query = """
    SELECT 
        *
    FROM demand_eda
    ORDER BY start_time;
"""

demand = pd.read_sql(query, conn)
demand["start_time"] = pd.to_datetime(demand["start_time"], utc = True)

conn.close()

In [3]:
fill_generation = generation.copy()

fill_generation = generation.sort_values(["fuel_type", "start_time"])
fill_generation["previous_generation"] = fill_generation.groupby("fuel_type")["generation_mw"].shift(1)
fill_generation["next_generation"] = fill_generation.groupby("fuel_type")["generation_mw"].shift(-1)
fill_generation["previous_start_time"] = fill_generation.groupby("fuel_type")["start_time"].shift(1)
fill_generation["interval"] = fill_generation["start_time"] - fill_generation["previous_start_time"]

wrong_intervals = fill_generation[fill_generation["interval"] == pd.Timedelta(hours = 1)].copy()
wrong_intervals["publish_time"] = wrong_intervals["start_time"]
wrong_intervals["start_time"] = wrong_intervals["start_time"] - pd.Timedelta(minutes = 30)
wrong_intervals["generation_mw"] = ((
    wrong_intervals["previous_generation"] + wrong_intervals["next_generation"]
) / 2).round(0)

wrong_intervals = wrong_intervals[["publish_time", "start_time", "fuel_type", "generation_mw"]]

generation = pd.concat([generation, wrong_intervals], ignore_index = True).sort_values("start_time")

In [4]:
fill_demand = demand.copy()

fill_demand["previous_demand"] = fill_demand["true_demand_mw"].shift(1)
fill_demand["next_demand"] = fill_demand["true_demand_mw"].shift(-1)
fill_demand["previous_start_time"] = fill_demand["start_time"].shift(1)
fill_demand["interval"] = fill_demand["start_time"] - fill_demand["previous_start_time"]

wrong_intervals = fill_demand[fill_demand["interval"] == pd.Timedelta(hours = 1)].copy()
wrong_intervals["start_time"] = wrong_intervals["start_time"] - pd.Timedelta(minutes = 30)
wrong_intervals["true_demand_mw"] = ((
    wrong_intervals["previous_demand"] + wrong_intervals["next_demand"]
) / 2).round(0)

wrong_intervals = wrong_intervals[["start_time", "true_demand_mw"]]

demand = pd.concat([demand, wrong_intervals], ignore_index = True).sort_values("start_time")
demand = demand.rename(columns = {"start_time": "prediction_time"})

In [5]:
demand["demand_lag_30m"] = demand["true_demand_mw"].shift(1)
demand["demand_lag_1h"] = demand["true_demand_mw"].shift(2)
demand["demand_lag_2h"] = demand["true_demand_mw"].shift(4)
demand["demand_lag_6h"] = demand["true_demand_mw"].shift(12)
demand["demand_lag_12h"] = demand["true_demand_mw"].shift(24)

demand["demand_rolling_3h"] = demand["true_demand_mw"].shift(1).rolling(6).mean().round(0)
demand["demand_rolling_6h"] = demand["true_demand_mw"].shift(1).rolling(12).mean().round(0)
demand["demand_rolling_12h"] = demand["true_demand_mw"].shift(1).rolling(24).mean().round(0)

In [6]:
horizon_dfs = []

for horizon in range(48):
    df = demand.copy()
    df["horizon"] = horizon + 1 
    df["target_time"] = df["prediction_time"] + pd.Timedelta(minutes = 30 * (horizon))

    horizon_dfs.append(df)

modelling = pd.concat(
    objs = horizon_dfs,
    ignore_index = True
)

target_demand = demand[["prediction_time", "true_demand_mw"]].rename(columns = {
    "true_demand_mw": "target_demand"
})

modelling = modelling.merge(
    right = target_demand,
    left_on = "target_time",
    right_on = "prediction_time",
    how = "left"
)

modelling = modelling.drop(columns = [
    "true_demand_mw",
    "prediction_time_y"
]).rename(columns = {
    "prediction_time_x": "reference_time"
})

modelling.head()

,reference_time,demand_lag_30m,demand_lag_1h,demand_lag_2h,demand_lag_6h,demand_lag_12h,demand_rolling_3h,demand_rolling_6h,demand_rolling_12h,horizon,target_time,target_demand
0,2023-08-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2023-08-01 00:00:00+00:00,17609.0
1,2023-08-01 00:30:00+00:00,17609.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2023-08-01 00:30:00+00:00,17596.0
2,2023-08-01 01:00:00+00:00,17596.0,17609.0,NaN,NaN,NaN,NaN,NaN,NaN,1,2023-08-01 01:00:00+00:00,17288.0
3,2023-08-01 01:30:00+00:00,17288.0,17596.0,NaN,NaN,NaN,NaN,NaN,NaN,1,2023-08-01 01:30:00+00:00,16797.0
4,2023-08-01 02:00:00+00:00,16797.0,17288.0,17609.0,NaN,NaN,NaN,NaN,NaN,1,2023-08-01 02:00:00+00:00,16742.0


In [7]:
demand_basic = demand[["prediction_time", "true_demand_mw"]].copy()

lag_times = {
    "24h": pd.Timedelta(hours = 24),
    "48h": pd.Timedelta(hours = 48),
    "7d": pd.Timedelta(days = 7)
}

for label, time in lag_times.items():
    lookup = demand_basic.rename(columns = {
        "prediction_time": "lag_time",
        "true_demand_mw": f"demand_lag_{label}"
    })

    modelling["lag_time"] = modelling["target_time"] - time

    modelling = modelling.merge(
        right = lookup,
        on = "lag_time",
        how = "left"
    ).drop(columns = "lag_time")

modelling = modelling.sort_values(["reference_time", "horizon"]).dropna().reset_index(drop = True)

modelling[[
    "reference_time", "target_time", "horizon", "target_demand",
    "demand_lag_24h", "demand_lag_48h", "demand_lag_7d"
]].head()

,reference_time,target_time,horizon,target_demand,demand_lag_24h,demand_lag_48h,demand_lag_7d
0,2023-08-07 00:30:00+00:00,2023-08-08 00:00:00+00:00,48,18379.0,17950.0,17272.0,17609.0
1,2023-08-07 01:00:00+00:00,2023-08-08 00:00:00+00:00,47,18379.0,17950.0,17272.0,17609.0
2,2023-08-07 01:00:00+00:00,2023-08-08 00:30:00+00:00,48,18102.0,17774.0,16922.0,17596.0
3,2023-08-07 01:30:00+00:00,2023-08-08 00:00:00+00:00,46,18379.0,17950.0,17272.0,17609.0
4,2023-08-07 01:30:00+00:00,2023-08-08 00:30:00+00:00,47,18102.0,17774.0,16922.0,17596.0


In [8]:
generation_pivot = generation.pivot_table(
    index = "publish_time",
    columns = "fuel_type",
    values = "generation_mw",
    aggfunc = "last"
).reset_index().sort_values("publish_time").drop(columns = "OIL")

generation_pivot.head()

fuel_type,publish_time,BIOMASS,CCGT,COAL,INTELEC,INTEW,INTFR,INTIFA2,INTIRL,INTNED,INTNEM,INTNSL,NPSHYD,NUCLEAR,OCGT,OTHER,PS,WIND
0,2023-08-01 00:30:00+00:00,1169.0,4459.0,0.0,644.0,-332.0,396.0,598.0,-372.0,-906.0,-154.0,338.0,145.0,2925.0,0.0,236.0,-306.0,9365.0
1,2023-08-01 01:00:00+00:00,1166.0,4618.0,0.0,646.0,-290.0,384.0,590.0,-372.0,-910.0,-150.0,274.0,146.0,2931.0,0.0,154.0,-360.0,9389.0
2,2023-08-01 01:30:00+00:00,1164.0,4625.0,0.0,696.0,-378.0,646.0,564.0,-372.0,-1010.0,98.0,20.0,146.0,2929.0,0.0,153.0,-358.0,9013.0
3,2023-08-01 02:00:00+00:00,1167.0,4337.0,0.0,698.0,-444.0,652.0,560.0,-372.0,-1010.0,102.0,-12.0,146.0,2928.0,0.0,155.0,-358.0,8960.0
4,2023-08-01 02:30:00+00:00,1169.0,4435.0,0.0,684.0,-480.0,578.0,496.0,-372.0,-1010.0,146.0,-58.0,145.0,2929.0,0.0,191.0,-492.0,8999.0


In [9]:
fuels = [
    "BIOMASS",
    "WIND",
    "PS",
    "OTHER",
    "OCGT",
    "NPSHYD",
    "NUCLEAR",
    "COAL",
    "CCGT"
]

interconnectors = [
    "INTNSL",
    "INTNEM",
    "INTIRL",
    "INTIFA2",
    "INTFR",
    "INTEW",
    "INTELEC",
    "INTNED"
]

generation_pivot["total_fuel"] = generation_pivot[fuels].sum(axis = 1)
generation_pivot["total_interconnector"] = generation_pivot[interconnectors].sum(axis = 1)
generation_pivot["total_generation"] = generation_pivot["total_fuel"] + generation_pivot["total_interconnector"]

generation_pivot["OTHER"] = np.log1p(generation_pivot["OTHER"])
generation_pivot["OCGT"] = np.log1p(generation_pivot["OCGT"])
generation_pivot["NPSHYD"] = np.log1p(generation_pivot["NPSHYD"])
generation_pivot["COAL"] = np.log1p(generation_pivot["COAL"])
generation_pivot["CCGT"] = np.log1p(generation_pivot["CCGT"])

generation_pivot["INTNSL"] = np.sign(generation_pivot["INTNSL"]) * np.log1p(np.abs(generation_pivot["INTNSL"]))
generation_pivot["INTNEM"] = np.sign(generation_pivot["INTNEM"]) * np.log1p(np.abs(generation_pivot["INTNEM"]))

features_for_lag = fuels + interconnectors + ["total_fuel", "total_interconnector", "total_generation"]

for feature in features_for_lag:
    generation_pivot[f"{feature}_lag_30m"] = generation_pivot[feature].shift(1)
    generation_pivot[f"{feature}_lag_1h"] = generation_pivot[feature].shift(2)
    generation_pivot[f"{feature}_lag_2h"] = generation_pivot[feature].shift(4)

generation_pivot = generation_pivot.rename(columns = {
    "publish_time": "reference_time"
})

generation_pivot[[
    "total_fuel", "total_interconnector", "total_generation",
    "OTHER_lag_30m", "OTHER_lag_1h", "OTHER_lag_2h"
]].head()

fuel_type,total_fuel,total_interconnector,total_generation,OTHER_lag_30m,OTHER_lag_1h,OTHER_lag_2h
0,17993.0,212.0,18205.0,NaN,NaN,NaN
1,18044.0,172.0,18216.0,5.468060,NaN,NaN
2,17672.0,264.0,17936.0,5.043425,5.468060,NaN
3,17335.0,174.0,17509.0,5.036953,5.043425,NaN
4,17376.0,-16.0,17360.0,5.049856,5.036953,5.46806


In [10]:
modelling = pd.merge_asof(
    left = modelling,
    right = generation_pivot,
    on = "reference_time",
    direction = "backward"
)

modelling.head()

,reference_time,demand_lag_30m,demand_lag_1h,demand_lag_2h,demand_lag_6h,demand_lag_12h,demand_rolling_3h,demand_rolling_6h,demand_rolling_12h,horizon,...,INTNED_lag_2h,total_fuel_lag_30m,total_fuel_lag_1h,total_fuel_lag_2h,total_interconnector_lag_30m,total_interconnector_lag_1h,total_interconnector_lag_2h,total_generation_lag_30m,total_generation_lag_1h,total_generation_lag_2h
0,2023-08-07 00:30:00+00:00,17950.0,18407.0,19622.0,24914.0,18764.0,19618.0,21984.0,21767.0,48,...,1002.0,12869.0,13338.0,15075.0,6158.0,6204.0,6250.0,19027.0,19542.0,21325.0
1,2023-08-07 01:00:00+00:00,17774.0,17950.0,18882.0,24745.0,18774.0,18891.0,21389.0,21725.0,47,...,1002.0,12976.0,12869.0,14006.0,5534.0,6158.0,6240.0,18510.0,19027.0,20246.0
2,2023-08-07 01:00:00+00:00,17774.0,17950.0,18882.0,24745.0,18774.0,18891.0,21389.0,21725.0,48,...,1002.0,12976.0,12869.0,14006.0,5534.0,6158.0,6240.0,18510.0,19027.0,20246.0
3,2023-08-07 01:30:00+00:00,17554.0,17774.0,18407.0,24684.0,18856.0,18365.0,20790.0,21674.0,46,...,808.0,12850.0,12976.0,13338.0,5510.0,5534.0,6204.0,18360.0,18510.0,19542.0
4,2023-08-07 01:30:00+00:00,17554.0,17774.0,18407.0,24684.0,18856.0,18365.0,20790.0,21674.0,47,...,808.0,12850.0,12976.0,13338.0,5510.0,5534.0,6204.0,18360.0,18510.0,19542.0


In [11]:
weather = weather.drop(columns = "apparent_temperature")

weather_pivot = weather.pivot(
    index = "forecast_time",
    columns = "location_name",
    values = [
        "temperature_2m",
        "relative_humidity_2m",
        "snowfall",
        "rain"
    ]
).reset_index()

weather_pivot.columns = [
    f"{weather_condition}_{city}".lower().replace(" ", "_")
    for weather_condition, city in weather_pivot.columns
]

weather_pivot.head()

,forecast_time_,temperature_2m_cardiff,temperature_2m_edinburgh,temperature_2m_inverness,temperature_2m_london,temperature_2m_manchester,temperature_2m_newcastle,temperature_2m_norwich,temperature_2m_plymouth,relative_humidity_2m_cardiff,...,snowfall_norwich,snowfall_plymouth,rain_cardiff,rain_edinburgh,rain_inverness,rain_london,rain_manchester,rain_newcastle,rain_norwich,rain_plymouth
0,2023-08-01 00:00:00+00:00,15.88,14.34,13.98,15.19,14.87,14.09,15.52,15.57,90.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.2,0.0,0.0,0.0
1,2023-08-01 01:00:00+00:00,15.58,14.29,13.58,14.79,14.67,13.89,15.32,15.57,87.0,...,0.0,0.0,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.0
2,2023-08-01 02:00:00+00:00,15.13,14.24,13.63,14.84,14.52,14.09,14.82,15.47,87.0,...,0.0,0.0,0.0,0.0,1.6,0.0,0.0,0.0,0.0,0.2
3,2023-08-01 03:00:00+00:00,14.93,14.04,13.58,15.24,14.52,14.09,15.07,15.52,83.0,...,0.0,0.0,0.0,0.3,0.9,0.0,0.0,0.0,0.0,0.0
4,2023-08-01 04:00:00+00:00,14.68,13.79,13.38,14.89,14.07,13.99,14.62,15.42,86.0,...,0.0,0.0,0.0,0.4,0.0,0.0,0.1,0.0,0.0,0.4


In [12]:
temperature_columns = [
    column for column in weather_pivot.columns
    if column.startswith("temp")
]

rain_columns = [
    col for col in weather_pivot.columns
    if col.startswith("rain")
]

snow_columns = [
    col for col in weather_pivot.columns
    if col.startswith("snow")
]

weather_pivot["temperature_mean"] = weather_pivot[temperature_columns].mean(axis = 1)
weather_pivot["temperature_min"] = weather_pivot[temperature_columns].min(axis = 1)
weather_pivot["temperature_max"] = weather_pivot[temperature_columns].max(axis = 1)
weather_pivot = weather_pivot.drop(columns = temperature_columns)

weather_pivot["cities_with_rain"] = (weather_pivot[rain_columns] > 0).sum(axis = 1)
weather_pivot["cities_with_snow"] = (weather_pivot[snow_columns] > 0).sum(axis = 1)

weather_pivot[rain_columns] = np.log1p(weather_pivot[rain_columns])
weather_pivot[snow_columns] = np.log1p(weather_pivot[snow_columns])

weather_pivot = weather_pivot.rename(columns = {
    "forecast_time_": "target_time"
})

weather_pivot[[
    "target_time", 
    "temperature_mean", "temperature_min", "temperature_max",
    "cities_with_rain", "cities_with_snow"
]].head()

,target_time,temperature_mean,temperature_min,temperature_max,cities_with_rain,cities_with_snow
0,2023-08-01 00:00:00+00:00,14.93000,13.98,15.88,2,0
1,2023-08-01 01:00:00+00:00,14.71125,13.58,15.58,1,0
2,2023-08-01 02:00:00+00:00,14.59250,13.63,15.47,2,0
3,2023-08-01 03:00:00+00:00,14.62375,13.58,15.52,2,0
4,2023-08-01 04:00:00+00:00,14.35500,13.38,15.42,3,0


In [13]:
modelling = modelling.sort_values("target_time").copy()

modelling = pd.merge_asof(
    left = modelling,
    right = weather_pivot,
    on = "target_time",
    direction = "backward"
)

modelling = modelling.sort_values(["reference_time", "horizon"]).reset_index(drop = True)

modelling.head()

,reference_time,demand_lag_30m,demand_lag_1h,demand_lag_2h,demand_lag_6h,demand_lag_12h,demand_rolling_3h,demand_rolling_6h,demand_rolling_12h,horizon,...,rain_london,rain_manchester,rain_newcastle,rain_norwich,rain_plymouth,temperature_mean,temperature_min,temperature_max,cities_with_rain,cities_with_snow
0,2023-08-07 00:30:00+00:00,17950.0,18407.0,19622.0,24914.0,18764.0,19618.0,21984.0,21767.0,48,...,0.0,0.0,0.0,0.0,0.0,13.255,11.23,14.82,0,0
1,2023-08-07 01:00:00+00:00,17774.0,17950.0,18882.0,24745.0,18774.0,18891.0,21389.0,21725.0,47,...,0.0,0.0,0.0,0.0,0.0,13.255,11.23,14.82,0,0
2,2023-08-07 01:00:00+00:00,17774.0,17950.0,18882.0,24745.0,18774.0,18891.0,21389.0,21725.0,48,...,0.0,0.0,0.0,0.0,0.0,13.255,11.23,14.82,0,0
3,2023-08-07 01:30:00+00:00,17554.0,17774.0,18407.0,24684.0,18856.0,18365.0,20790.0,21674.0,46,...,0.0,0.0,0.0,0.0,0.0,13.255,11.23,14.82,0,0
4,2023-08-07 01:30:00+00:00,17554.0,17774.0,18407.0,24684.0,18856.0,18365.0,20790.0,21674.0,47,...,0.0,0.0,0.0,0.0,0.0,13.255,11.23,14.82,0,0


In [14]:
def months_to_seasons(month):
    if 3 <= month < 6:
        return "Spring"
    elif 6 <= month < 9:
        return "Summer"
    elif 9 <= month < 12:
        return "Autumn"
    else:
        return "Winter"

modelling["month"] = modelling["target_time"].dt.month
modelling["season"] = modelling["target_time"].dt.month.apply(months_to_seasons)

modelling["hour"] = modelling["target_time"].dt.hour
modelling["minute"] = modelling["target_time"].dt.minute
modelling["time_of_day"] = modelling["hour"] + modelling["minute"] / 60
modelling = modelling.drop(columns = [
    "hour",
    "minute"
])

modelling["is_weekend"] = (modelling["target_time"].dt.dayofweek >= 5)

modelling[[
    "reference_time", "target_time", "horizon", "target_demand", 
    "month", "season", "time_of_day", "is_weekend"
]].head()

,reference_time,target_time,horizon,target_demand,month,season,time_of_day,is_weekend
0,2023-08-07 00:30:00+00:00,2023-08-08 00:00:00+00:00,48,18379.0,8,Summer,0.0,False
1,2023-08-07 01:00:00+00:00,2023-08-08 00:00:00+00:00,47,18379.0,8,Summer,0.0,False
2,2023-08-07 01:00:00+00:00,2023-08-08 00:30:00+00:00,48,18102.0,8,Summer,0.5,False
3,2023-08-07 01:30:00+00:00,2023-08-08 00:00:00+00:00,46,18379.0,8,Summer,0.0,False
4,2023-08-07 01:30:00+00:00,2023-08-08 00:30:00+00:00,47,18102.0,8,Summer,0.5,False


In [15]:
modelling.info()

<class 'pandas.DataFrame'>
RangeIndex: 2509056 entries, 0 to 2509055
Data columns (total 128 columns):
 #    Column                           Dtype              
---   ------                           -----              
 0    reference_time                   datetime64[us, UTC]
 1    demand_lag_30m                   float64            
 2    demand_lag_1h                    float64            
 3    demand_lag_2h                    float64            
 4    demand_lag_6h                    float64            
 5    demand_lag_12h                   float64            
 6    demand_rolling_3h                float64            
 7    demand_rolling_6h                float64            
 8    demand_rolling_12h               float64            
 9    horizon                          int64              
 10   target_time                      datetime64[us, UTC]
 11   target_demand                    float64            
 12   demand_lag_24h                   float64            
 13   demand